## Surrogate vs. Thermochemical library
**Computational speedup**

Import libraries

In [1]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import dataframe_image as dfi

In [2]:
import utils

Define inputs

In [3]:
# "csv" header
header = ["t", "mean", "std"]
# Paths to timings
prefix = "/home/zanardi/Codes/ML/PyCOMET/run/noneq/RVE45/adiabatic/postproc/speedup/core/"
# Timings
methods = {
  "ode": {
    "integrators": ("sundials",), #("lsode", "sundials"),
    "max_orders": np.arange(2,6).astype(str).tolist(),
    "rtol_orders": np.arange(6,10).astype(str).tolist()
  },
  "sur": {
    "models": ("merged_f32",), #("merged_f32", "merged_f64"),
    "devices": ("cpu", "gpu")
  }
}

In [4]:
path_to_saving = os.path.abspath("./../output/speedup/")
os.makedirs(path_to_saving, exist_ok=True)

Read timings

In [5]:
def read_timings(filename):
  timings = pd.read_csv(filename)
  timings.columns = header
  return timings

In [6]:
timingsf = {}
for method in methods.keys():
  timingsf[method] = {}
  if (method == "ode"):
    for integrator in methods[method]["integrators"]:
      timingsf[method][integrator] = {}
      for rtol in methods[method]["rtol_orders"]:
        krtol = "rtol"+rtol
        timingsf[method][integrator][krtol] = {}
        for max_order in methods[method]["max_orders"]:
          kord = "ord"+max_order
          filename = prefix + f"/{method}/data/{integrator}/timings_rtol{rtol}_ord{max_order}.csv"
          timingsf[method][integrator][krtol][kord] = filename
  elif (method == "sur"):
    for model in methods[method]["models"]:
      timingsf[method][model] = {}
      for device in methods[method]["devices"]:
        filename = prefix + f"/{method}/data/timings_{model}_{device}.csv"
        timingsf[method][model][device] = filename

In [7]:
timings = tf.nest.map_structure(read_timings, timingsf)

Compute speedup

In [8]:
def compute_speedup(tode, tsur):
  # Extract mean and std
  values = {}
  for k in ("mean", "std"):
    values[k] = np.vstack([df[k].to_numpy().squeeze() for df in (tode, tsur)])
  # Compute speedup
  rmean, rstd = utils.ratio_normal_approx(values["mean"], values["std"])
  # Compose data
  time = tode["t"].to_numpy().squeeze()
  data = np.vstack([time, rmean, rstd]).T
  return pd.DataFrame(data=data, columns=header)

In [9]:
speedup = {}
for model in methods["sur"]["models"]:
  speedup[model] = {}
  for device in methods["sur"]["devices"]:
    tsur = timings["sur"][model][device]
    speedup[model][device] = tf.nest.map_structure(
      lambda tode: compute_speedup(tode, tsur), timings["ode"]
    )

In [10]:
speedup_flat = pd.json_normalize(speedup, sep="_").to_dict(orient="records")[0]

Save speedup

In [11]:
def sformat(mean, std):
  return "{:.3f} ".format(mean) + u"\u00B1" + " {:.3f}".format(std)

In [12]:
def save_speedup(df, filename, max_order, savefig=False):
  # Save original data
  df.to_csv(
    path_or_buf=filename + ".csv",
    float_format="%.8e",
    index=False
  )
  # Print original data in a table
  if savefig:
    svalues = []
    sdata = df.to_dict(orient="list")
    for i in range(len(sdata["t"])):
      svalues.append(sformat(sdata["mean"][i], sdata["std"][i]))
    sdata = {
      "t [s]": ["{:.0e}".format(ti) for ti in sdata["t"]], 
      "Speedup": svalues
    }
    dfs = pd.DataFrame.from_dict(sdata)
    dfs_styled = dfs.style.set_caption(f"Max order {max_order}") \
      .hide(axis="index").background_gradient()
    dfi.export(dfs_styled, filename + "_table.png")

In [13]:
ipath = path_to_saving + "/data/"
os.makedirs(ipath, exist_ok=True)
for (ispeedup, idata) in speedup_flat.items():
  save_speedup(
    df=idata,
    filename=ipath + ispeedup,
    max_order=max_order
  )

In [14]:
speedup_ratio = {}
for model in methods["sur"]["models"]:
  speedup_ratio[model] = tf.nest.map_structure(
    lambda x, y: (x["mean"]/y["mean"]).to_numpy().tolist(),
    speedup[model]["gpu"],
    speedup[model]["cpu"]
  )

with open(ipath + "/speedup_ratio_gpu_cpu.json", "w") as file:
  json.dump(speedup_ratio, file, indent=2)

Plot speedup

In [15]:
ipath = path_to_saving + "/figs/"
os.makedirs(ipath, exist_ok=True)

In [16]:
for idata in speedup_flat.values():
  time = idata["t"].to_numpy().squeeze()
  break

> 1D plot

In [17]:
for model in methods["sur"]["models"]:
  for device in methods["sur"]["devices"]:
    for integrator in methods["ode"]["integrators"]:
      for rtol in methods["ode"]["rtol_orders"]:
        krtol = "rtol"+rtol
        ispeedup = speedup[model][device][integrator][krtol]
        idata = {"x": time, "mean": None, "std": None}
        for k in ("mean",): # ("mean", "std"):
          idata[k] = np.vstack([df[k].values for df in ispeedup.values()])
        utils.plot_speedup_1d(
          **idata,
          var_names=[f"BDF{o[-1:]}" for o in ispeedup.keys()],
          labels=["$t_f$ [s]", "Speedup"],
          scales=["log", "log"],
          figname=ipath + f"/1d_{model}_{device}_{integrator}_{krtol}",
          save=True,
          show=False,
          extensions=("pdf",)
        )

> 2D plot

In [18]:
xlbl = [r"${0:s}$".format(utils.as_si(ti)) for ti in time]
ylbl = [10**(-int(rtol)) for rtol in methods["ode"]["rtol_orders"]]
ylbl = [r"${0:s}$".format(utils.as_si(yi)) for yi in ylbl]

In [19]:
for model in methods["sur"]["models"]:
  for device in methods["sur"]["devices"]:
    for integrator in methods["ode"]["integrators"]:
      ispeedup = speedup[model][device][integrator]
      for max_order in methods["ode"]["max_orders"]:
        kord = "ord"+max_order
        idata = np.vstack([df[kord]["mean"].values for df in ispeedup.values()])
        utils.plot_speedup_2d(
          xlbl,
          ylbl,
          z=idata,
          figname=ipath + f"/2d_{model}_{device}_{integrator}_{kord}",
          save=True,
          show=False,
          extensions=("pdf",)
        )